In [ ]:
import sys
import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Có CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1),
        "GB"
    )
else:
    print("Chưa được cấp GPU.")

!nvidia-smi
!df -h /content

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import json
import sys
import torch
from datetime import datetime, timezone

PROJECT = Path("/content/drive/MyDrive/helmet_project")

for folder in [
    "notebooks", "configs", "data", "runs",
    "results/dataset_audit", "results/evaluation",
    "results/speed", "results/error_analysis",
    "demo", "report"
]:
    (PROJECT / folder).mkdir(parents=True, exist_ok=True)

environment = {
    "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "pytorch": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "vram_gib": round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ),
}

env_path = PROJECT / "configs" / "environment_initial.json"
env_path.write_text(
    json.dumps(environment, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Thư mục dự án:", PROJECT)
print("Đã lưu thông tin môi trường:", env_path)
print("Kiểm tra đọc lại:", json.loads(env_path.read_text())["gpu"])

In [ ]:
%pip -q install gdown

In [ ]:
from pathlib import Path
import gdown

RAW = Path("/content/helmet_raw")
RAW.mkdir(parents=True, exist_ok=True)

archive_path = RAW / "SHWD_download"

downloaded = gdown.download(
    id="1qWm7rrwvjAWs1slymbrLaCf7Q-wnGLEX",
    output=str(archive_path),
    quiet=False,
)

if downloaded is None:
    raise RuntimeError("Chưa tải được dataset. Dừng tại đây để kiểm tra.")

size_mb = archive_path.stat().st_size / 1024**2
print(f"Đã tải: {archive_path}")
print(f"Dung lượng: {size_mb:.1f} MB")

# Xác định loại file trước khi giải nén
!file /content/helmet_raw/SHWD_download

In [ ]:
from pathlib import Path
from collections import Counter
import zipfile
import xml.etree.ElementTree as ET

archive = Path("/content/helmet_raw/SHWD_download")
extract_dir = Path("/content/helmet_raw/extracted")
extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(archive) as z:
    # Kiểm tra đường dẫn trước khi giải nén
    root = extract_dir.resolve()
    for member in z.infolist():
        target = (root / member.filename).resolve()
        if not target.is_relative_to(root):
            raise ValueError(f"Đường dẫn ZIP không hợp lệ: {member.filename}")

    print("Đang giải nén...")
    z.extractall(extract_dir)

image_extensions = {".jpg", ".jpeg", ".png", ".bmp"}
images = sorted(
    p for p in extract_dir.rglob("*")
    if p.suffix.lower() in image_extensions
)
xml_files = sorted(extract_dir.rglob("*.xml"))

class_counts = Counter()
xml_errors = []

for path in xml_files:
    try:
        tree = ET.parse(path)
        for obj in tree.getroot().findall("object"):
            class_counts[obj.findtext("name", default="").strip()] += 1
    except (ET.ParseError, OSError) as error:
        xml_errors.append((str(path), str(error)))

print("\nSố file ảnh:", len(images))
print("Số file XML:", len(xml_files))
print("Số box theo tên lớp:", dict(class_counts))
print("Số XML không đọc được:", len(xml_errors))

print("\nThư mục chứa ảnh:")
for folder in sorted({str(p.parent) for p in images}):
    print(folder)

print("\nThư mục chứa nhãn:")
for folder in sorted({str(p.parent) for p in xml_files}):
    print(folder)

print("\nCác file chia tập có sẵn:")
for p in sorted(extract_dir.rglob("*.txt")):
    if "ImageSets" in p.parts:
        print(p.relative_to(extract_dir))

if xml_errors:
    print("\nVí dụ lỗi XML:", xml_errors[:3])

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET
import pandas as pd
from PIL import Image

BASE = Path("/content/helmet_raw/extracted/VOC2028")
IMG, ANN, SETS = BASE/"JPEGImages", BASE/"Annotations", BASE/"ImageSets/Main"
OUT = Path("/content/drive/MyDrive/helmet_project/results/dataset_audit")
OUT.mkdir(parents=True, exist_ok=True)

# 1) Các tập chia sẵn
splits = {n: [l.strip() for l in (SETS/f"{n}.txt").read_text().splitlines() if l.strip()]
          for n in ["train", "val", "test", "trainval"]}
for n, ids in splits.items():
    print(f"{n}: {len(ids)} (duy nhất: {len(set(ids))})")
tr, va, te = (set(splits[k]) for k in ["train", "val", "test"])
print("train∩val:", len(tr & va), "| train∩test:", len(tr & te), "| val∩test:", len(va & te))
print("trainval == train ∪ val:", set(splits["trainval"]) == (tr | va))

stem2img = {p.stem: p for p in IMG.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}}
in_splits = tr | va | te
print("Ảnh không thuộc tập nào:", len(set(stem2img) - in_splits))
print("ID trong tập nhưng thiếu ảnh:", len(in_splits - set(stem2img)))

# 2) Đọc nhãn
rows, no_obj, size_mismatch = [], [], 0
for xp in sorted(ANN.glob("*.xml")):
    root = ET.parse(xp).getroot()
    W = int(float(root.findtext("size/width", "0")))
    H = int(float(root.findtext("size/height", "0")))
    imgp = stem2img.get(xp.stem)
    if imgp is not None:
        with Image.open(imgp) as im:
            if im.size != (W, H):
                size_mismatch += 1
                W, H = im.size          # dùng kích thước thật của ảnh
    objs = root.findall("object")
    if not objs:
        no_obj.append(xp.stem)
    for o in objs:
        bb = o.find("bndbox")
        x1, y1, x2, y2 = (float(bb.findtext(k)) for k in ("xmin", "ymin", "xmax", "ymax"))
        rows.append(dict(id=xp.stem, cls=o.findtext("name").strip(), W=W, H=H,
                         x1=x1, y1=y1, x2=x2, y2=y2, difficult=o.findtext("difficult")))
df = pd.DataFrame(rows)
df["split"] = df["id"].map(lambda i: "train" if i in tr else "val" if i in va else "test" if i in te else "none")
df["bw"], df["bh"] = df.x2 - df.x1, df.y2 - df.y1
df["long_px_640"] = df[["bw", "bh"]].max(axis=1) * 640 / df[["W", "H"]].max(axis=1)

print("\nẢnh không có box:", len(no_obj), "| XML lệch kích thước so với ảnh thật:", size_mismatch)
print("Box lỗi (x2<=x1 hoặc y2<=y1):", int(((df.bw <= 0) | (df.bh <= 0)).sum()))
print("Box ra ngoài ảnh:", int(((df.x1 < 0) | (df.y1 < 0) | (df.x2 > df.W) | (df.y2 > df.H)).sum()))
print("Giá trị difficult:", df.difficult.value_counts(dropna=False).to_dict())

# 3) Thống kê theo tập và lớp
print("\nSố box theo tập và lớp:")
print(pd.crosstab(df.split, df.cls, margins=True))
print("\nSố ảnh có ít nhất một box của lớp, theo tập:")
print(df.groupby(["split", "cls"])["id"].nunique().unstack(fill_value=0))
per_img = df.groupby("id").size()
print("\nSố box mỗi ảnh:\n", per_img.describe().round(1))
print("5 ảnh đông nhất:", per_img.sort_values(ascending=False).head(5).to_dict())
print("\nKích thước cạnh dài của box (quy về ảnh 640 px), theo lớp:")
print(df.groupby("cls")["long_px_640"].describe(percentiles=[.1, .5, .9]).round(1))

print("\nẢnh chứa dog:", df[df.cls == "dog"]["id"].tolist())
df.to_csv(OUT / "boxes_raw.csv", index=False)
print("Đã lưu:", OUT / "boxes_raw.csv")

In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import ImageDraw

random.seed(0)
COLORS = {"hat": "lime", "person": "red", "dog": "cyan"}
by_id = df.groupby("id")
dog_ids = df[df.cls == "dog"]["id"].unique().tolist()
crowded = per_img.sort_values(ascending=False).head(3).index.tolist()
hat_few_person = [i for i, g in by_id if (g.cls == "hat").any() and (g.cls == "person").sum() <= 2]
all_ids_list = sorted(stem2img)
picks = (dog_ids[:3] + crowded +
         random.sample(hat_few_person, min(4, len(hat_few_person))) +
         random.sample(all_ids_list, 8))[:18]

def draw(i):
    im = Image.open(stem2img[i]).convert("RGB")
    d = ImageDraw.Draw(im)
    lw = max(2, im.width // 300)
    for _, r in by_id.get_group(i).iterrows():
        d.rectangle([r.x1, r.y1, r.x2, r.y2], outline=COLORS.get(r.cls, "yellow"), width=lw)
    return im

fig, axes = plt.subplots(3, 6, figsize=(24, 12))
for ax, i in zip(axes.ravel(), picks):
    ax.imshow(draw(i)); ax.set_title(f"{i} | {len(by_id.get_group(i))} box", fontsize=9); ax.axis("off")
plt.tight_layout()
plt.savefig(OUT / "sample_labels.png", dpi=110)
plt.show()

In [ ]:
%pip -q install imagehash
import re, numpy as np, imagehash
from tqdm import tqdm

# 1) Nguồn ảnh suy từ tiền tố ID (PartA_, PartB_, part2_, hoặc toàn số)
def source(i):
    m = re.match(r"^([A-Za-z0-9]+)_", i)
    return m.group(1) if m else "numeric"
df["source"] = df["id"].map(source)
print("Số ảnh theo nguồn và tập:")
print(df.drop_duplicates("id").groupby(["source", "split"]).size().unstack(fill_value=0))
print("\nSố box theo nguồn và lớp:")
print(pd.crosstab(df.source, df.cls))

# 2) Mức nhỏ của box theo lớp (quy về ảnh 640 px)
for c in ["hat", "person"]:
    s = df[df.cls == c]["long_px_640"]
    print(f"\n{c}: <4px={(s<4).mean():.1%} | <16px={(s<16).mean():.1%} | <32px={(s<32).mean():.1%}")
print("\ndifficult theo lớp:\n", pd.crosstab(df.cls, df.difficult))

# 3) Perceptual hash cho mọi ảnh
ids_all = sorted(stem2img)
hashes = {}
for i in tqdm(ids_all):
    with Image.open(stem2img[i]) as im:
        hashes[i] = np.uint64(int(str(imagehash.phash(im)), 16))

POP = np.array([bin(k).count("1") for k in range(256)], dtype=np.uint8)
def hamming(arr, h):
    x = np.bitwise_xor(arr, h)
    return POP[x.view(np.uint8).reshape(-1, 8)].sum(1)

def cross(a_ids, b_ids, thr=4):
    B = np.array([hashes[i] for i in b_ids], dtype=np.uint64)
    out = []
    for i in a_ids:
        d = hamming(B, hashes[i])
        for j in np.where(d <= thr)[0]:
            out.append((i, b_ids[j], int(d[j])))
    return out

tr_l, va_l, te_l = sorted(tr), sorted(va), sorted(te)
tt = cross(te_l, tr_l)
vt = cross(va_l, tr_l)
print(f"\nCặp test~train (Hamming<=4): {len(tt)} | ảnh test bị ảnh hưởng: {len({a for a,_,_ in tt})}/{len(te_l)}")
print(f"Cặp val~train  (Hamming<=4): {len(vt)} | ảnh val bị ảnh hưởng:  {len({a for a,_,_ in vt})}/{len(va_l)}")
print("Trùng hệt (d=0) test~train:", sum(1 for *_, d in tt if d == 0))
print("Theo nguồn của ảnh test bị ảnh hưởng:",
      pd.Series([source(a) for a, _, _ in tt]).value_counts().to_dict())

# 4) Xem 8 cặp test~train gần giống nhất và ảnh 000377
show = sorted(tt, key=lambda t: t[2])[:8]
fig, axes = plt.subplots(2, 8, figsize=(32, 8))
for k, (a, b, d) in enumerate(show):
    axes[0, k].imshow(Image.open(stem2img[a])); axes[0, k].set_title(f"test {a} (d={d})", fontsize=9)
    axes[1, k].imshow(Image.open(stem2img[b])); axes[1, k].set_title(f"train {b}", fontsize=9)
for ax in axes.ravel(): ax.axis("off")
plt.tight_layout(); plt.savefig(OUT / "near_duplicates.png", dpi=90); plt.show()

plt.figure(figsize=(8, 8)); plt.imshow(draw("000377")); plt.axis("off"); plt.show()

In [ ]:
import numpy as np, pandas as pd
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/helmet_project")
MAN = PROJECT / "data" / "manifests"; MAN.mkdir(parents=True, exist_ok=True)
src = {i: source(i) for i in ids_all}

def nearest(a_ids, b_ids):
    B = np.array([hashes[i] for i in b_ids], dtype=np.uint64)
    rec = []
    for i in a_ids:
        d = hamming(B, hashes[i]); k = int(d.argmin())
        rec.append((i, int(d[k]), b_ids[k]))
    return pd.DataFrame(rec, columns=["id", "d", "nn"]).set_index("id")

nt = nearest(te_l, sorted(tr | va))     # test so với train+val
nv = nearest(va_l, tr_l)                # val so với train

# 1) Độ nhạy theo ngưỡng Hamming
for name, nn_df in [("test ~ (train+val)", nt), ("val ~ train", nv)]:
    rows = []
    for thr in [0, 2, 4, 6, 8, 10]:
        flag = nn_df["d"] <= thr
        row = {"thr": thr, "tổng": int(flag.sum())}
        row.update(flag.groupby(nn_df.index.map(src)).sum().astype(int).to_dict())
        rows.append(row)
    print(f"\n{name}: số ảnh có bản gần giống theo ngưỡng")
    print(pd.DataFrame(rows).set_index("thr").fillna(0).astype(int))

# 2) Xem các cặp ở khoảng cách 5-6 để kiểm tra ngưỡng 6
cand = nt[(nt.d >= 5) & (nt.d <= 6)]
pick = cand.sample(min(6, len(cand)), random_state=0) if len(cand) else cand
if len(pick):
    fig, axes = plt.subplots(2, len(pick), figsize=(4 * len(pick), 7), squeeze=False)
    for k, (i, r) in enumerate(pick.iterrows()):
        axes[0, k].imshow(Image.open(stem2img[i])); axes[0, k].set_title(f"test {i} d={r.d}", fontsize=9)
        axes[1, k].imshow(Image.open(stem2img[r.nn])); axes[1, k].set_title(f"gần nhất {r.nn}", fontsize=9)
    for ax in axes.ravel(): ax.axis("off")
    plt.tight_layout(); plt.show()

# 3) Tạo danh sách sạch và lưu manifests
THR = 6
test_clean = [i for i in te_l if nt.loc[i, "d"] > THR]
val_clean  = [i for i in va_l if nv.loc[i, "d"] > THR]
for name, ids in {"train": tr_l, "val": va_l, "test": te_l,
                  "val_clean": val_clean, "test_clean": test_clean}.items():
    (MAN / f"{name}.txt").write_text("\n".join(ids))
nt.to_csv(MAN / "nearest_test_vs_trainval.csv"); nv.to_csv(MAN / "nearest_val_vs_train.csv")
print(f"\nNgưỡng={THR}: test {len(te_l)} -> {len(test_clean)} | val {len(va_l)} -> {len(val_clean)}")

for name, ids in [("train", tr_l), ("val_clean", val_clean), ("test_clean", test_clean)]:
    sub = df[df.id.isin(ids)]
    print(f"\n{name}: {len(ids)} ảnh"); print(pd.crosstab(sub.source, sub.cls))
site = df[df.source.isin(["numeric", "part2"]) & (df.cls == "person")]
print("\nẢnh có 'person' trong nhóm ảnh có mũ, theo tập gốc:", site.groupby("split")["id"].nunique().to_dict())

# 4) Xem 3 khung 'dog'
dg = df[df.cls == "dog"]
im = Image.open(stem2img["000377"]).convert("RGB")
fig, ax = plt.subplots(1, len(dg), figsize=(4 * len(dg), 4), squeeze=False)
for a, (_, r) in zip(ax[0], dg.iterrows()):
    a.imshow(im.crop((int(r.x1), int(r.y1), int(r.x2), int(r.y2)))); a.axis("off")
    a.set_title(f"dog, difficult={r.difficult}")
plt.show()

In [ ]:
import shutil, yaml
from pathlib import Path

YOLO = Path("/content/helmet_yolo")
(YOLO/"images").mkdir(parents=True, exist_ok=True)
(YOLO/"labels").mkdir(parents=True, exist_ok=True)
CFG = PROJECT/"configs"; MAN = PROJECT/"data"/"manifests"
CFG.mkdir(parents=True, exist_ok=True); MAN.mkdir(parents=True, exist_ok=True)

# 1) dog -> hat (3 box, đã xem crop: đều là đầu đội mũ)
d2 = df.copy()
d2.loc[d2.cls == "dog", "cls"] = "hat"
CLASS_ID = {"hat": 0, "person": 1}
NAMES = {0: "helmet", 1: "bare_head"}

# 2) Chuyển sang YOLO
for i, g in d2.groupby("id"):
    dst = YOLO/"images"/stem2img[i].name
    if not dst.exists():
        shutil.copy2(stem2img[i], dst)
    lines = [f"{CLASS_ID[r.cls]} {(r.x1+r.x2)/2/r.W:.6f} {(r.y1+r.y2)/2/r.H:.6f} "
             f"{(r.x2-r.x1)/r.W:.6f} {(r.y2-r.y1)/r.H:.6f}" for r in g.itertuples()]
    (YOLO/"labels"/f"{i}.txt").write_text("\n".join(lines) + "\n")

# 3) Danh sách ảnh theo tập và data.yaml
SITE = {"numeric", "part2"}
lists = {"train": tr_l, "val": va_l, "test": te_l,
         "val_clean": val_clean, "test_clean": test_clean,
         "val_clean_site": [i for i in val_clean if source(i) in SITE],
         "test_clean_site": [i for i in test_clean if source(i) in SITE]}
for name, ids in lists.items():
    (YOLO/f"{name}.txt").write_text("\n".join(str(YOLO/"images"/stem2img[i].name) for i in ids) + "\n")
    (MAN/f"{name}.txt").write_text("\n".join(ids))

def write_yaml(fname, tr, va, te):
    cfg = {"path": str(YOLO), "train": f"{tr}.txt", "val": f"{va}.txt", "test": f"{te}.txt", "names": NAMES}
    (YOLO/fname).write_text(yaml.safe_dump(cfg, sort_keys=False))
    shutil.copy(YOLO/fname, CFG/fname)
write_yaml("data.yaml", "train", "val_clean", "test_clean")
write_yaml("data_test_official.yaml", "train", "val_clean", "test")
write_yaml("data_test_clean_site.yaml", "train", "val_clean_site", "test_clean_site")

# 4) Kiểm tra
cnt, bad = {0: 0, 1: 0}, 0
for p in (YOLO/"labels").glob("*.txt"):
    for ln in p.read_text().splitlines():
        t = ln.split()
        if len(t) != 5 or int(t[0]) not in cnt or any(not (0 <= float(x) <= 1) for x in t[1:]):
            bad += 1
        else:
            cnt[int(t[0])] += 1
print("Ảnh:", len(list((YOLO/'images').iterdir())), "| file nhãn:", len(list((YOLO/'labels').glob('*.txt'))), "| dòng lỗi:", bad)
print("Box theo lớp (mong đợi {0: 9047, 1: 111514}):", cnt)
print("Đường dẫn ảnh không tồn tại:",
      sum(not Path(l).exists() for n in lists for l in (YOLO/f"{n}.txt").read_text().split()))
for name, ids in lists.items():
    sub = d2[d2.id.isin(ids)]
    print(f"{name:16s} {len(ids):5d} ảnh | hat={(sub.cls=='hat').sum():6d} person={(sub.cls=='person').sum():6d}")

In [ ]:
import json, hashlib
sha = hashlib.sha256()
with open("/content/helmet_raw/SHWD_download", "rb") as f:
    for chunk in iter(lambda: f.read(1 << 24), b""):
        sha.update(chunk)
(CFG/"dataset_source.json").write_text(json.dumps({
    "name": "SHWD (VOC2028)", "downloaded": "2026-11-03",
    "gdrive_id": "1qWm7rrwvjAWs1slymbrLaCf7Q-wnGLEX", "sha256": sha.hexdigest(),
    "license_checked": False}, indent=2))
(CFG/"label_rules.md").write_text(
    "- helmet (id 0): khung quanh mũ bảo hộ/đầu đội mũ; 3 box 'dog' đã đổi thành helmet.\n"
    "- bare_head (id 1): khung quanh đầu không đội mũ, không phải cả người.\n"
    "- Lọc trùng: pHash, Hamming <= 6; val_clean/test_clean loại ảnh gần giống train.\n")
print("Đã lưu")

In [ ]:
%pip -q install ultralytics
import sys, subprocess, ultralytics
from ultralytics import YOLO as UL
print("Ultralytics:", ultralytics.__version__)
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
(CFG/"pip_freeze.txt").write_text(freeze)

m = UL("yolov8n.pt")
img = str(YOLO/"images"/stem2img["000163"].name)
r = m.predict(img, conf=0.25, save=True,
              project=str(PROJECT/"results"/"smoke"), name="yolov8n_coco", exist_ok=True)
print("Số phát hiện:", len(r[0].boxes), "| lưu tại:", r[0].save_dir)

In [ ]:
import time, shutil
from ultralytics import YOLO as UL

SMOKE = Path("/content/runs_smoke")
t0 = time.time()
m = UL("yolov8n.pt")
m.train(data=str(YOLO/"data.yaml"), epochs=3, imgsz=640, batch=16,
        seed=42, project=str(SMOKE), name="smoke_v8n", exist_ok=True)
print(f"\nTổng thời gian: {(time.time()-t0)/60:.1f} phút")

run = SMOKE/"smoke_v8n"
print("File trong thư mục run:", sorted(p.name for p in run.iterdir()))
print("File trọng số:", sorted(p.name for p in (run/"weights").iterdir()))
print(pd.read_csv(run/"results.csv").round(4).T.to_string())
shutil.copytree(run, PROJECT/"runs"/"smoke_v8n", dirs_exist_ok=True)
print("Đã chép lên Drive:", PROJECT/"runs"/"smoke_v8n")

In [ ]:
import time, torch, pandas as pd
from ultralytics import YOLO as UL

def probe(weights, frac=0.25, epochs=2, batch=16):
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    name = "probe_" + weights.replace(".pt", "")
    m = UL(weights)
    m.train(data=str(YOLO/"data.yaml"), epochs=epochs, imgsz=640, batch=batch,
            fraction=frac, seed=42, project=str(SMOKE), name=name,
            exist_ok=True, plots=False)
    res = pd.read_csv(SMOKE/name/"results.csv")
    ep2 = res["time"].iloc[-1] - res["time"].iloc[-2]
    return dict(model=weights, batch=batch,
                peak_gb=round(torch.cuda.max_memory_reserved()/1024**3, 1),
                epoch2_s=round(ep2), est_full_epoch_s=round(ep2/frac),
                est_50ep_h=round(ep2/frac*50/3600, 1))

rows = []
for w in ["yolov8n.pt", "yolov8s.pt", "yolov8m.pt"]:
    try:
        rows.append(probe(w))
    except Exception as e:
        rows.append(dict(model=w, batch=16, lỗi=str(e)[:200]))
    print(rows[-1])
tbl = pd.DataFrame(rows)
print(tbl.to_string(index=False))
tbl.to_csv(PROJECT/"results"/"smoke"/"probe_timing.csv", index=False)

In [ ]:
import pandas as pd
from ultralytics import YOLO as UL

# 1) Nạp lại mô hình đã lưu trên Drive và dự đoán 2 ảnh val_clean_site
saved = PROJECT/"runs"/"smoke_v8n"/"weights"/"best.pt"
m = UL(str(saved))
imgs = [str(YOLO/"images"/stem2img[i].name) for i in lists["val_clean_site"][:2]]
r = m.predict(imgs, conf=0.25, save=True,
              project=str(PROJECT/"results"/"smoke"), name="reload_check", exist_ok=True)
for res in r:
    print(Path(res.path).name, "->", {m.names[int(c)]: int((res.boxes.cls == c).sum()) for c in res.boxes.cls.unique()})

# 2) Giả lập bị ngắt sau epoch 1, rồi học tiếp từ last.pt
def stop_after_epoch_1(trainer):
    if trainer.epoch == 0:
        raise RuntimeError("Dừng giả lập sau epoch 1 (để thử học tiếp)")

t = UL("yolov8n.pt")
t.add_callback("on_fit_epoch_end", stop_after_epoch_1)
try:
    t.train(data=str(YOLO/"data.yaml"), epochs=3, imgsz=640, batch=16, fraction=0.1,
            seed=42, project=str(SMOKE), name="resume_test", exist_ok=True, plots=False)
except RuntimeError as e:
    print("\nĐã dừng:", e)

last = SMOKE/"resume_test"/"weights"/"last.pt"
print("last.pt tồn tại:", last.exists())
UL(str(last)).train(resume=True)
print(pd.read_csv(SMOKE/"resume_test"/"results.csv")[["epoch","metrics/mAP50(B)"]])

## Khôi phục dữ liệu

In [ ]:
import sys, subprocess, shutil, zipfile, yaml, re
from pathlib import Path
import xml.etree.ElementTree as ET
import pandas as pd, torch

assert torch.cuda.is_available(), 'Chưa có GPU: Runtime > Change runtime type > T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/helmet_project')
MAN = PROJECT/'data'/'manifests'
RAW, YOLO = Path('/content/helmet_raw'), Path('/content/helmet_yolo')

def source(i):
    m = re.match(r'^([A-Za-z0-9]+)_', i)
    return m.group(1) if m else 'numeric'

def build_dataset(RAW, MAN, YOLO):
    RAW.mkdir(parents=True, exist_ok=True)
    archive, extract_dir = RAW/'SHWD_download', RAW/'extracted'
    if not archive.exists():
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'], check=True)
        import gdown
        gdown.download(id='1qWm7rrwvjAWs1slymbrLaCf7Q-wnGLEX', output=str(archive), quiet=False)
    if not extract_dir.exists():
        with zipfile.ZipFile(archive) as z:
            root = extract_dir.resolve(); extract_dir.mkdir()
            for mb in z.infolist():
                assert (root/mb.filename).resolve().is_relative_to(root), mb.filename
            z.extractall(extract_dir)
    BASE = extract_dir/'VOC2028'
    stem2img = {p.stem: p for p in (BASE/'JPEGImages').iterdir()
                if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}}
    rows = []
    for xp in sorted((BASE/'Annotations').glob('*.xml')):
        r = ET.parse(xp).getroot()
        W = int(float(r.findtext('size/width', '0'))); H = int(float(r.findtext('size/height', '0')))
        for o in r.findall('object'):
            bb = o.find('bndbox')
            x1, y1, x2, y2 = (float(bb.findtext(k)) for k in ('xmin', 'ymin', 'xmax', 'ymax'))
            rows.append(dict(id=xp.stem, cls=o.findtext('name').strip(), W=W, H=H, x1=x1, y1=y1, x2=x2, y2=y2))
    d2 = pd.DataFrame(rows)
    d2.loc[d2.cls == 'dog', 'cls'] = 'hat'
    CLASS_ID, NAMES = {'hat': 0, 'person': 1}, {0: 'helmet', 1: 'bare_head'}
    rd = lambda n: [l for l in (MAN/f'{n}.txt').read_text().split() if l]
    tr_l, va_l, te_l = rd('train'), rd('val'), rd('test')
    val_clean, test_clean = rd('val_clean'), rd('test_clean')
    SITE = {'numeric', 'part2'}
    lists = {'train': tr_l, 'val': va_l, 'test': te_l, 'val_clean': val_clean, 'test_clean': test_clean,
             'val_clean_site': [i for i in val_clean if source(i) in SITE],
             'test_clean_site': [i for i in test_clean if source(i) in SITE]}
    (YOLO/'images').mkdir(parents=True, exist_ok=True); (YOLO/'labels').mkdir(parents=True, exist_ok=True)
    for i, g in d2.groupby('id'):
        dst = YOLO/'images'/stem2img[i].name
        if not dst.exists(): shutil.copy2(stem2img[i], dst)
        (YOLO/'labels'/f'{i}.txt').write_text('\n'.join(
            f'{CLASS_ID[r.cls]} {(r.x1+r.x2)/2/r.W:.6f} {(r.y1+r.y2)/2/r.H:.6f} '
            f'{(r.x2-r.x1)/r.W:.6f} {(r.y2-r.y1)/r.H:.6f}' for r in g.itertuples()) + '\n')
    for name, ids in lists.items():
        (YOLO/f'{name}.txt').write_text('\n'.join(str(YOLO/'images'/stem2img[i].name) for i in ids) + '\n')
    def write_yaml(fname, tr, va, te):
        (YOLO/fname).write_text(yaml.safe_dump({'path': str(YOLO), 'train': f'{tr}.txt', 'val': f'{va}.txt',
                                                'test': f'{te}.txt', 'names': NAMES}, sort_keys=False))
    write_yaml('data.yaml', 'train', 'val_clean', 'test_clean')
    write_yaml('data_test_official.yaml', 'train', 'val_clean', 'test')
    write_yaml('data_test_clean_site.yaml', 'train', 'val_clean_site', 'test_clean_site')
    cnt = d2.cls.map(CLASS_ID).value_counts().to_dict()
    sizes = [len(v) for v in lists.values()]
    print('Box theo lớp (mong đợi {0: 9047, 1: 111514}):', cnt)
    print('Số ảnh mỗi tập (mong đợi 5457/607/1517/425/1087/264/628):', sizes)
    assert cnt == {0: 9047, 1: 111514} and sizes == [5457, 607, 1517, 425, 1087, 264, 628]
    print('Dựng bộ dữ liệu xong, khớp số liệu.')
    return d2, lists, stem2img

d2, lists, stem2img = build_dataset(RAW, MAN, YOLO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.202'], check=True)
import ultralytics; print('Ultralytics:', ultralytics.__version__)

In [ ]:
import json, torch, ultralytics
from datetime import datetime, timezone
from ultralytics import YOLO as UL

EPOCHS = 30          # 50 nếu đủ units, ngược lại 30; PHẢI giống nhau cho mọi mô hình
cfg = {
  'saved_at_utc': datetime.now(timezone.utc).isoformat(),
  'ultralytics': ultralytics.__version__, 'torch': torch.__version__,
  'gpu': torch.cuda.get_device_name(0),
  'data_yaml': str(YOLO/'data.yaml'),
  'imgsz': 640, 'batch': 16, 'seed': 42, 'epochs': EPOCHS, 'device': 0, 'save_period': 5,
  'optimizer': 'auto; ghi giá trị thực từ args.yaml sau mỗi run (lần thử: AdamW, lr0 0.001667)',
  'order': ['v8m_main', 'v8s_main', 'v8n_main', 'v5su_main (nếu chạy baseline)'],
  'selection': 'best.pt do thư viện chọn trên val_clean; không dùng test để chọn',
  'eval_sets': ['test gốc (ghi rõ rò rỉ)', 'test_clean', 'test_clean_site'],
  'speed_protocol': 'mọi mô hình cùng thiết bị và phiên, batch 1, warm-up, đo lặp, lưu thống kê',
}
(PROJECT/'configs'/'experiment_config.json').write_text(json.dumps(cfg, indent=2, ensure_ascii=False))
print('Đã lưu cấu hình:', cfg['ultralytics'], cfg['gpu'], 'epochs =', EPOCHS)

try:
    b = UL('yolov5su.pt'); b.info()
    print('Tải được yolov5su.pt')
except Exception as e:
    print('KHÔNG tải được yolov5su.pt:', str(e)[:300])

In [ ]:
import time, json
import pandas as pd
from ultralytics import YOLO as UL

EPOCHS, BATCH, SEED, IMGSZ = 30, 16, 42, 640    # PHẢI giống ô 1
WORK = PROJECT/'runs'

def run_main(weights, name):
    last = WORK/name/'weights'/'last.pt'
    t0 = time.time()
    if last.exists():
        print('Có last.pt, học tiếp:', last)
        UL(str(last)).train(resume=True)
    else:
        UL(weights).train(data=str(YOLO/'data.yaml'), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
                          seed=SEED, device=0, save_period=5, plots=True,
                          project=str(WORK), name=name, exist_ok=True)
    hours = (time.time() - t0) / 3600
    res = pd.read_csv(WORK/name/'results.csv')
    print(name, 'xong sau', round(hours, 2), 'giờ trong phiên này | số epoch:', len(res))
    (WORK/name/'run_note.json').write_text(json.dumps({'hours_this_session': hours, 'epochs_in_csv': len(res)}))

run_main('yolov8m.pt', 'v8m_main')

In [ ]:
from pathlib import Path
import pandas as pd
import yaml

PROJECT = Path("/content/drive/MyDrive/helmet_project")
RUN_M = PROJECT / "runs" / "v8m_main"

required = [
    "weights/best.pt",
    "weights/last.pt",
    "results.csv",
    "args.yaml",
]

for name in required:
    path = RUN_M / name
    assert path.is_file() and path.stat().st_size > 0, f"Thiếu/rỗng: {path}"
    print(f"OK: {name} — {path.stat().st_size / 1024:.1f} KB")

results_m = pd.read_csv(RUN_M / "results.csv")
results_m.columns = results_m.columns.str.strip()

assert len(results_m) == 30, f"Cần kiểm tra số dòng: {len(results_m)}"
assert int(results_m["epoch"].iloc[-1]) == 30

args_m = yaml.safe_load((RUN_M / "args.yaml").read_text())

print("\nĐã xác nhận đủ kết quả 30 epochs.")
print("Cấu hình:")
for key in ["model", "data", "epochs", "imgsz", "batch", "seed",
            "optimizer", "lr0", "fraction"]:
    print(f"{key}: {args_m.get(key)}")

In [ ]:
import time
import torch
import ultralytics
from ultralytics import YOLO

assert torch.cuda.is_available(), "Chưa có GPU."
assert ultralytics.__version__ == "8.4.202", (
    f"Phiên bản hiện tại là {ultralytics.__version__}; "
    "cần giữ 8.4.202 như run v8m."
)

RUN_S = PROJECT / "runs" / "v8s_main"
assert not RUN_S.exists(), (
    f"{RUN_S} đã tồn tại. Dừng để kiểm tra trước khi chạy lại."
)

train_cfg = dict(args_m)

# Thông tin riêng của lần chạy trước
for key in ["model", "task", "mode", "save_dir", "resume"]:
    train_cfg.pop(key, None)

train_cfg.update(
    project=str(PROJECT / "runs"),
    name="v8s_main",
    exist_ok=False,
    resume=False,
    device=0,
)

assert train_cfg["epochs"] == 30
assert train_cfg["batch"] == 16
assert train_cfg["imgsz"] == 640
assert train_cfg["seed"] == 42
assert train_cfg.get("fraction", 1.0) == 1.0
assert Path(train_cfg["data"]).is_file(), "Thiếu data.yaml."

print("GPU:", torch.cuda.get_device_name(0))
print("Bắt đầu YOLOv8s — 30 epochs.")
print("Nơi lưu:", RUN_S)

started = time.monotonic()
model_s = YOLO("yolov8s.pt")
model_s.train(**train_cfg)

print(f"\nĐã chạy xong sau {(time.monotonic() - started) / 60:.1f} phút.")
print("Kết quả:", RUN_S)

In [ ]:
# Verify completed runs, save audit, then train YOLOv8n with the same protocol.
from pathlib import Path
import json, time, yaml
from datetime import datetime, timezone
import pandas as pd
import torch, ultralytics
from ultralytics import YOLO as UL

PROJECT = Path('/content/drive/MyDrive/helmet_project')
assert torch.cuda.is_available(), 'GPU unavailable'
assert ultralytics.__version__ == '8.4.202', ultralytics.__version__
base = yaml.safe_load((PROJECT/'runs/v8m_main/args.yaml').read_text())
audit = {}
for run_name in ['v8m_main', 'v8s_main']:
    run = PROJECT/'runs'/run_name
    for relative in ['weights/best.pt', 'weights/last.pt', 'results.csv', 'args.yaml']:
        p = run/relative
        assert p.is_file() and p.stat().st_size > 0, str(p)
    table = pd.read_csv(run/'results.csv')
    table.columns = table.columns.str.strip()
    assert table['epoch'].astype(int).tolist() == list(range(1,31)), run_name
    args = yaml.safe_load((run/'args.yaml').read_text())
    for key in ['data','epochs','imgsz','batch','seed','fraction','optimizer','lr0','close_mosaic','amp','deterministic']:
        assert args.get(key) == base.get(key), (run_name,key)
    # Confirm best checkpoint can actually be loaded, without evaluating test.
    loaded = UL(str(run/'weights/best.pt'))
    assert loaded.names == {0:'helmet',1:'bare_head'}, loaded.names
    audit[run_name] = {'epochs':30, 'files_verified':True, 'best_load_verified':True,
        'best_validation_map50_95_csv':float(table['metrics/mAP50-95(B)'].max())}
    print('VERIFIED:', run_name, audit[run_name])
    del loaded

stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
(PROJECT/'results').mkdir(exist_ok=True)
(PROJECT/'results'/f'run_audit_{stamp}.json').write_text(json.dumps(audit,indent=2))
run_n = PROJECT/'runs/v8n_main'
assert not run_n.exists(), 'v8n_main already exists: inspect before any retry'
config = dict(base)
for key in ['model','task','mode','save_dir','resume']:
    config.pop(key,None)
config.update(project=str(PROJECT/'runs'),name='v8n_main',exist_ok=False,resume=False,device=0)
assert config['epochs']==30 and config['batch']==16 and config['imgsz']==640 and config['seed']==42
assert config.get('fraction',1.0)==1.0
assert Path(config['data']).is_file(), config['data']
started_at = datetime.now(timezone.utc).isoformat()
record = {'run':'v8n_main','model':'yolov8n.pt','status':'starting','started_at_utc':started_at,
          'ultralytics':ultralytics.__version__,'gpu':torch.cuda.get_device_name(0),'config':config}
record_path = PROJECT/'configs'/f'v8n_main_execution_{stamp}.json'
record_path.write_text(json.dumps(record,indent=2,ensure_ascii=False))
torch.cuda.empty_cache()
print('STARTING v8n_main: 30 epochs; validation only; outputs on Drive', flush=True)
t0 = time.monotonic()
try:
    model_n = UL('yolov8n.pt')
    model_n.train(**config)
except Exception as error:
    record.update(status='error',error=repr(error),elapsed_seconds=time.monotonic()-t0)
    record_path.write_text(json.dumps(record,indent=2,ensure_ascii=False))
    raise
record.update(status='completed',elapsed_seconds=time.monotonic()-t0)
record_path.write_text(json.dumps(record,indent=2,ensure_ascii=False))
print('COMPLETED v8n_main:', record['elapsed_seconds']/60, 'minutes')


In [ ]:
# Audit YOLOv8n and start the planned YOLOv5su baseline without changing protocol.
from pathlib import Path
import json, time, yaml, gc
from datetime import datetime, timezone
import pandas as pd
import torch, ultralytics
from ultralytics import YOLO as UL
PROJECT = Path('/content/drive/MyDrive/helmet_project')
assert torch.cuda.is_available(), 'GPU unavailable'
assert ultralytics.__version__ == '8.4.202'
base = yaml.safe_load((PROJECT/'runs/v8m_main/args.yaml').read_text())
run = PROJECT/'runs/v8n_main'
audit = {'run':'v8n_main','files':{}}
for relative in ['weights/best.pt','weights/last.pt','results.csv','args.yaml']:
    p = run/relative
    assert p.is_file() and p.stat().st_size > 0, str(p)
    audit['files'][relative] = p.stat().st_size
table = pd.read_csv(run/'results.csv')
table.columns = table.columns.str.strip()
assert table['epoch'].astype(int).tolist() == list(range(1,31))
args = yaml.safe_load((run/'args.yaml').read_text())
for key in ['data','epochs','imgsz','batch','seed','fraction','optimizer','lr0','close_mosaic','amp','deterministic']:
    assert args.get(key) == base.get(key), key
loaded = UL(str(run/'weights/best.pt'))
assert loaded.names == {0:'helmet',1:'bare_head'}
audit.update(epochs=30,best_load_verified=True,best_validation_map50_95_csv=float(table['metrics/mAP50-95(B)'].max()),optimizer_observed='AdamW',initial_lr_observed=0.001667,momentum_observed=0.9)
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
(PROJECT/'results'/f'v8n_audit_{stamp}.json').write_text(json.dumps(audit,indent=2))
print('VERIFIED v8n_main:',audit,flush=True)
del loaded
if 'model_n' in globals():
    del model_n
gc.collect()
torch.cuda.empty_cache()
assert not (PROJECT/'runs/v5su_main').exists(), 'v5su_main already exists; inspect before retry'
config = dict(base)
for key in ['model','task','mode','save_dir','resume']:
    config.pop(key,None)
config.update(project=str(PROJECT/'runs'),name='v5su_main',exist_ok=False,resume=False,device=0)
assert config['epochs']==30 and config['batch']==16 and config['imgsz']==640 and config['seed']==42
assert config.get('fraction',1.0)==1.0 and Path(config['data']).is_file()
record = {'run':'v5su_main','model':'yolov5su.pt','status':'starting','started_at_utc':datetime.now(timezone.utc).isoformat(),'ultralytics':ultralytics.__version__,'gpu':torch.cuda.get_device_name(0),'config':config}
record_path = PROJECT/'configs'/f'v5su_main_execution_{stamp}.json'
record_path.write_text(json.dumps(record,indent=2,ensure_ascii=False))
t0 = time.monotonic()
print('STARTING v5su_main: 30 epochs; same train and validation; outputs on Drive',flush=True)
try:
    model_baseline = UL('yolov5su.pt')
    model_baseline.train(**config)
except Exception as error:
    record.update(status='error',error=repr(error),elapsed_seconds=time.monotonic()-t0)
    record_path.write_text(json.dumps(record,indent=2,ensure_ascii=False))
    raise
record.update(status='completed',elapsed_seconds=time.monotonic()-t0)
record_path.write_text(json.dumps(record,indent=2,ensure_ascii=False))
print('COMPLETED v5su_main:',record['elapsed_seconds']/60,'minutes')
